# Fine-tuning do Modelo Moirai-MoE com Componentes Bayesianos para Criptomoedas

Este notebook demonstra como fazer o fine-tuning do modelo Moirai-MoE com componentes bayesianos para previsão de séries temporais de criptomoedas usando o repositório uni2ts.

## Passos deste notebook:
1. Configuração do ambiente Kaggle
2. Download e instalação do repositório uni2ts
3. Preparação dos dados de criptomoedas
4. Configuração e ajuste do modelo Moirai-MoE
5. Treinamento e monitoramento do modelo
6. Avaliação do modelo e visualização de resultados
7. Salvamento do modelo e exportação

## 1. Configuração do Ambiente Kaggle

Primeiro, vamos verificar a versão do Python e configurar o ambiente Kaggle. O Moirai-MoE é otimizado para Python 3.11, então é importante verificar a compatibilidade.

In [ ]:
import sys
print(f"Python version: {sys.version}")

# Verificar disponibilidade de GPU (recomendado para treinamento)
!nvidia-smi

### 1.1 Configuração de Diretórios

Vamos criar a estrutura de diretórios necessária para o projeto.

In [ ]:
import os

# Diretórios principais
BASE_DIR = "/kaggle/working"
REPO_DIR = os.path.join(BASE_DIR, "uni2ts")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
LOG_DIR = os.path.join(BASE_DIR, "logs")
PLOT_DIR = os.path.join(BASE_DIR, "plots")

# Criar diretórios
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Diretórios criados:")
print(f"- Repositório: {REPO_DIR}")
print(f"- Dados: {DATA_DIR}")
print(f"- Saída: {OUTPUT_DIR}")
print(f"- Logs: {LOG_DIR}")
print(f"- Gráficos: {PLOT_DIR}")

## 2. Download e Instalação do Repositório uni2ts

Agora vamos clonar o repositório e instalar o pacote e suas dependências.

In [ ]:
# Clonar o repositório
!git clone https://github.com/waldefran/uni2ts.git {REPO_DIR}

# Mudar para o diretório do repositório
%cd {REPO_DIR}

# Listar conteúdo do diretório para confirmar
!ls -la

### 2.1 Instalação das Dependências

Vamos instalar as dependências necessárias para o treinamento do modelo.

In [ ]:
# Instalar o pacote em modo de desenvolvimento e suas dependências
!pip install -e .

# Instalar dependências específicas para criptomoedas
!pip install -r requirements_crypto.txt

# Verificar instalação
!pip list | grep -E "torch|lightning|pandas|numpy|scipy|matplotlib|wandb"

### 2.2 Verificação do Ambiente

Vamos verificar se o ambiente está configurado corretamente para o treinamento do modelo bayesiano de criptomoedas.

In [ ]:
# Executar o script de verificação do ambiente
!python environment_report.py

### 2.3 Verificação dos Componentes SOTA

Vamos verificar se todos os componentes necessários para o pipeline SOTA estão funcionando corretamente.

In [ ]:
# Testar a importação dos componentes principais
import torch
import pytorch_lightning as pl
from uni2ts.model.crypto.bayesian_head import BayesianPredictionHead
from uni2ts.loss.bayesian_elbo import BayesianELBOLoss
from uni2ts.data.builder.crypto import CryptoDatasetBuilder
from uni2ts.callbacks.bayesian_uncertainty import BayesianUncertaintyMonitor
from uni2ts.callbacks.elbo_annealing import ELBOAnnealingCallback
from uni2ts.model.moirai import MoiraiMoEModule

print("Importação dos componentes SOTA realizada com sucesso!")

## 3. Preparação dos Dados de Criptomoedas

Agora vamos preparar os dados de criptomoedas para o treinamento do modelo. Para isso, utilizaremos o CryptoDatasetBuilder do uni2ts.

### 3.1 Download dos Dados da Binance

Primeiro, vamos baixar os dados da Binance usando o script `binanceDataloader.py` do repositório.

In [ ]:
# Criar diretório para dados da Binance
BINANCE_DATA_DIR = os.path.join(DATA_DIR, "binance_data")
os.makedirs(BINANCE_DATA_DIR, exist_ok=True)

# Configuração para o download dos dados
# Definindo um período relativamente curto para os testes
# Ajuste conforme necessário para seus experimentos
start_date = "2023-01-01"
end_date = "2023-06-30"

# Lista de ativos a serem baixados
assets = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT"]
interval = "1h"  # Intervalo de 1 hora

# Executar o script de download para cada ativo
for asset in assets:
    print(f"Baixando dados para {asset}...")
    !python binanceDataloader.py --symbol {asset} --interval {interval} \
        --start {start_date} --end {end_date} \
        --output_dir {BINANCE_DATA_DIR}

# Listar os arquivos baixados
print("\nArquivos baixados:")
!ls -la {BINANCE_DATA_DIR}

### 3.2 Preparação do Dataset

Agora vamos preparar o dataset para o treinamento usando o CryptoDatasetBuilder. Vamos carregar o arquivo de configuração e fazer os ajustes necessários.

In [ ]:
# Importar bibliotecas necessárias
import yaml
import copy
from pathlib import Path

# Carregar o arquivo de configuração
config_path = os.path.join(REPO_DIR, "configs", "crypto", "finetune_bayesian_moe.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Ajustar caminhos no arquivo de configuração
config["data"]["dataset"]["args"]["data_dir"] = BINANCE_DATA_DIR
config["data"]["dataset"]["args"]["target_assets"] = assets

# Ajustar parâmetros de treinamento
config["trainer"]["max_epochs"] = 10  # Reduzido para este exemplo, aumente para treinamento real
config["trainer"]["default_root_dir"] = OUTPUT_DIR

# Ajustar callbacks
for callback in config["callbacks"]:
    if callback["class"] == "BayesianUncertaintyMonitor":
        callback["args"]["plot_dir"] = PLOT_DIR

# Exibir a configuração atualizada
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Criar o CryptoDatasetBuilder
from uni2ts.data.builder.crypto import CryptoDatasetBuilder

dataset_config = config["data"]["dataset"]
dataset_builder = CryptoDatasetBuilder(**dataset_config["args"])

# Preparar os dados
train_dataset, val_dataset, test_dataset = dataset_builder.build_datasets()

print(f"Tamanho do dataset de treino: {len(train_dataset)}")
print(f"Tamanho do dataset de validação: {len(val_dataset)}")
print(f"Tamanho do dataset de teste: {len(test_dataset)}")

### 3.3 Criação dos DataLoaders

Agora vamos criar os DataLoaders para o treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
from torch.utils.data import DataLoader

# Parâmetros para os DataLoaders
batch_size = config["data"]["dataloader"]["args"]["batch_size"]
num_workers = config["data"]["dataloader"]["args"].get("num_workers", 4)

# Criar os DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

print(f"DataLoaders criados com batch size = {batch_size} e num_workers = {num_workers}")

### 3.4 Análise Exploratória dos Dados

Vamos analisar brevemente os dados para entender melhor a estrutura e características dos nossos dados de treinamento.

In [ ]:
# Importar bibliotecas de visualização
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Obter uma amostra do dataset
sample_batch = next(iter(train_loader))

# Extrair informações
print(f"Chaves do batch: {list(sample_batch.keys())}")
print(f"Formato dos dados de entrada (x): {sample_batch['x'].shape}")
print(f"Formato dos dados de saída (y): {sample_batch['y'].shape}")

# Extrair e visualizar uma série temporal de exemplo
example_idx = 0
example_x = sample_batch['x'][example_idx].numpy()
example_y = sample_batch['y'][example_idx].numpy()

# Determinar o número de features
n_features = example_x.shape[1]

# Plotar cada feature
fig, axs = plt.subplots(n_features, 1, figsize=(10, 3*n_features))
for i in range(n_features):
    if n_features > 1:
        ax = axs[i]
    else:
        ax = axs
    ax.plot(example_x[:, i], label='Entrada')
    # Adicionar valores de predição (y) se for a primeira feature (assumindo que é o target)
    if i == 0:
        # Criando um array do tamanho da entrada com NaNs
        full_series = np.full(len(example_x), np.nan)
        # Adicionando os valores de y no final
        full_series[-len(example_y):] = example_y
        ax.plot(range(len(example_x) - len(example_y), len(example_x)), example_y, 'r-', label='Alvo (y)')
    ax.set_title(f'Feature {i}')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Configuração e Ajuste do Modelo Moirai-MoE

Agora vamos configurar o modelo Moirai-MoE com componentes bayesianos para o fine-tuning.

### 4.1 Configuração do Modelo

Vamos criar o modelo Moirai-MoE com a configuração do arquivo YAML.

In [ ]:
# Importar bibliotecas necessárias
from uni2ts.model.moirai import MoiraiMoEModule
from uni2ts.loss.bayesian_elbo import BayesianELBOLoss
import torch

# Verificar disponibilidade de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

# Configuração do modelo
model_config = config["model"]
model_args = model_config["args"]

# Obter dimensões do dataset
sample_batch = next(iter(train_loader))
input_dim = sample_batch["x"].shape[2]  # Número de features de entrada
context_length = sample_batch["x"].shape[1]  # Comprimento da sequência de contexto
target_dim = sample_batch["y"].shape[1]  # Número de features a serem previstas

print(f"Dimensões do input: {input_dim}")
print(f"Comprimento do contexto: {context_length}")
print(f"Dimensões do target: {target_dim}")

# Ajustar configurações do modelo com base nos dados
model_args["in_dim"] = input_dim
model_args["target_dim"] = target_dim

# Criar o modelo
model = MoiraiMoEModule(**model_args)

# Configuração da função de perda
loss_config = config["loss"]
loss_args = loss_config["args"]
loss_fn = BayesianELBOLoss(**loss_args)

# Configurar o otimizador
optim_config = config["optimizer"]
optimizer = getattr(torch.optim, optim_config["class"])(model.parameters(), **optim_config["args"])

# Resumo do modelo
print("\nResumo do Modelo:")
print(f"Classe do modelo: {model.__class__.__name__}")
print(f"Número total de parâmetros: {sum(p.numel() for p in model.parameters())}")
print(f"Função de perda: {loss_fn.__class__.__name__}")
print(f"Otimizador: {optimizer.__class__.__name__}")

### 4.2 Configuração de Callbacks

Vamos configurar os callbacks para monitorar o treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from uni2ts.callbacks.bayesian_uncertainty import BayesianUncertaintyMonitor
from uni2ts.callbacks.elbo_annealing import ELBOAnnealingCallback

# Lista para armazenar os callbacks
callbacks = []

# Adicionar callbacks a partir da configuração
for callback_config in config["callbacks"]:
    callback_class_name = callback_config["class"]
    callback_args = callback_config.get("args", {})
    
    if callback_class_name == "ModelCheckpoint":
        callback = ModelCheckpoint(
            dirpath=os.path.join(OUTPUT_DIR, "checkpoints"),
            **callback_args
        )
    elif callback_class_name == "EarlyStopping":
        callback = EarlyStopping(**callback_args)
    elif callback_class_name == "LearningRateMonitor":
        callback = LearningRateMonitor(**callback_args)
    elif callback_class_name == "BayesianUncertaintyMonitor":
        callback = BayesianUncertaintyMonitor(**callback_args)
    elif callback_class_name == "ELBOAnnealingCallback":
        callback = ELBOAnnealingCallback(**callback_args)
    else:
        print(f"Callback não reconhecido: {callback_class_name}")
        continue
    
    callbacks.append(callback)
    print(f"Callback adicionado: {callback_class_name}")

print(f"\nTotal de callbacks configurados: {len(callbacks)}")

### 4.3 Configuração do Logger

Vamos configurar o logger para monitorar métricas durante o treinamento.

In [ ]:
# Importar bibliotecas necessárias
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger

# Configurar CSVLogger
csv_logger = CSVLogger(
    save_dir=LOG_DIR,
    name="crypto_finetune",
    version=None
)

# Configurar TensorBoardLogger
tb_logger = TensorBoardLogger(
    save_dir=LOG_DIR,
    name="crypto_finetune_tb",
    version=None
)

# Lista de loggers
loggers = [csv_logger, tb_logger]

# Adicionar WandbLogger se desejado (opcional)
use_wandb = False
if use_wandb:
    try:
        from pytorch_lightning.loggers import WandbLogger
        import wandb
        
        # Inicializar WandbLogger
        wandb_logger = WandbLogger(
            project="uni2ts-crypto",
            name="finetune-bayesian-moe",
            save_dir=LOG_DIR
        )
        loggers.append(wandb_logger)
        print("WandbLogger configurado com sucesso!")
    except ImportError:
        print("WandbLogger não pôde ser configurado. Pacote wandb não encontrado.")

print(f"\nTotal de loggers configurados: {len(loggers)}")

## 5. Treinamento e Monitoramento do Modelo

Agora vamos treinar o modelo usando o PyTorch Lightning.

### 5.1 Configuração do Trainer

Vamos configurar o Trainer do PyTorch Lightning.

In [ ]:
# Importar bibliotecas necessárias
import pytorch_lightning as pl

# Configuração do Trainer
trainer_config = config["trainer"]

# Adicionar configurações específicas do Kaggle
trainer_config["accelerator"] = "gpu" if torch.cuda.is_available() else "cpu"
trainer_config["devices"] = 1
trainer_config["default_root_dir"] = OUTPUT_DIR

# Criar o Trainer
trainer = pl.Trainer(
    logger=loggers,
    callbacks=callbacks,
    **trainer_config
)

print(f"Trainer configurado com acelerador: {trainer_config['accelerator']}")
print(f"Número máximo de épocas: {trainer_config['max_epochs']}")

### 5.2 Criação do Lightning Module

Vamos criar o Lightning Module para treinamento do modelo.

In [ ]:
# Importar bibliotecas necessárias
import pytorch_lightning as pl
import sys
from pathlib import Path

# Garantir que scripts/ esteja no path para importação
scripts_path = os.path.join(REPO_DIR, "scripts")
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

# Importar o módulo Lightning já existente no projeto em vez de redefini-lo
from crypto.finetune_model import MoiraiBayesianLightningModule

# Criar configuração para o módulo Lightning
lightning_config = {
    "model": {
        "pretrained_model_name_or_path": model_config["args"].get("pretrained_model_name_or_path", ""),
        "prediction_length": model_args.get("prediction_length", 24),
        "context_length": context_length,
        "patch_size": model_args.get("patch_size", 1),
        "num_samples": model_args.get("num_samples", 100),
        "prediction_head": {
            "_target_": "uni2ts.model.crypto.bayesian_head.BayesianPredictionHead",
            "input_size": model_args.get("d_model", 512),
            "hidden_size": model_args.get("d_model", 512),
            "output_size": target_dim,
            "dropout": model_args.get("dropout", 0.1),
            "prior_scale": loss_args.get("prior_scale", 1.0),
        },
        "freeze_backbone": model_args.get("freeze_backbone", False)
    },
    "loss_func": {
        "_target_": "uni2ts.loss.bayesian_elbo.BayesianELBOLoss",
        "kl_weight": loss_args.get("kl_weight", 1.0),
        "reduction": loss_args.get("reduction", "mean")
    },
    "optimizer": {
        "lr": optim_config["args"].get("lr", 1e-4),
        "weight_decay": optim_config["args"].get("weight_decay", 1e-5)
    }
}

# Criar o Lightning Module usando a classe existente no projeto
pl_module = MoiraiBayesianLightningModule(lightning_config)

print("Lightning Module configurado com sucesso usando a classe existente MoiraiBayesianLightningModule!")
print(f"Isso garante alinhamento com o pipeline SOTA do projeto uni2ts.")

### 5.3 Treinamento do Modelo

Agora vamos treinar o modelo com os dados de criptomoedas.

In [ ]:
# Preparar dados no formato esperado pelo módulo Lightning
# Converter os PyTorch DataLoaders para o formato esperado pelo MoiraiBayesianLightningModule
def convert_batch_format(batch):
    """Converter formato de batch se necessário"""
    if 'x' in batch and 'y' in batch:
        # Converter do formato do notebook para o formato esperado pelo modelo
        return {
            'past_target': batch['x'],
            'past_observed_target': torch.ones_like(batch['x']),
            'future_target': batch['y']
        }
    return batch

class BatchConverterDataLoader:
    def __init__(self, dataloader):
        self.dataloader = dataloader
        
    def __iter__(self):
        for batch in self.dataloader:
            yield convert_batch_format(batch)
            
    def __len__(self):
        return len(self.dataloader)

# Envolver os DataLoaders para garantir compatibilidade
train_loader_wrapped = BatchConverterDataLoader(train_loader)
val_loader_wrapped = BatchConverterDataLoader(val_loader)

# Treinar o modelo
print("Iniciando treinamento...")
trainer.fit(pl_module, train_loader_wrapped, val_loader_wrapped)
print("Treinamento concluído!")

### 5.4 Avaliação do Modelo no Conjunto de Teste

In [ ]:
# Envolver o dataloader de teste com o conversor
test_loader_wrapped = BatchConverterDataLoader(test_loader)

# Avaliar o modelo no conjunto de teste
print("Avaliando modelo no conjunto de teste...")
test_results = trainer.test(pl_module, test_loader_wrapped)
print(f"Resultados do teste: {test_results}")

## 6. Avaliação do Modelo e Visualização de Resultados

Vamos visualizar as métricas e resultados do treinamento.

### 6.1 Visualização de Métricas de Treinamento

In [ ]:
# Importar bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar as métricas do treinamento
metrics_path = os.path.join(LOG_DIR, "crypto_finetune", "version_0", "metrics.csv")
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    
    # Filtrar métricas relevantes
    train_loss = metrics_df[metrics_df['train/loss_epoch'].notna()][['epoch', 'train/loss_epoch']]
    val_loss = metrics_df[metrics_df['val/loss'].notna()][['epoch', 'val/loss']]
    
    # Configurar o plot
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=train_loss, x='epoch', y='train/loss_epoch', label='Train Loss')
    sns.lineplot(data=val_loss, x='epoch', y='val/loss', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (ELBO)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print(f"Arquivo de métricas não encontrado em {metrics_path}")

### 6.2 Visualização de Predições com Incerteza

Vamos visualizar algumas predições do modelo com intervalos de confiança.

In [ ]:
# Importar bibliotecas necessárias
import torch
from uni2ts.distribution.student_t import StudentT
import numpy as np
import matplotlib.pyplot as plt

# Função para visualizar predições com intervalos de confiança
def visualize_predictions_with_uncertainty(model, dataloader, num_samples=5):
    model.eval()
    samples = []
    with torch.no_grad():
        for batch in dataloader:
            if len(samples) >= num_samples:
                break
                
            # Converter formato do batch se necessário
            if not isinstance(batch, dict) or ('x' in batch and 'y' in batch):
                x = batch['x']
                y = batch['y']
                # Converter para formato esperado pelo modelo
                model_batch = {
                    'past_target': x,
                    'past_observed_target': torch.ones_like(x),
                    'future_target': y
                }
            else:
                model_batch = batch
                x = batch.get('past_target', batch.get('target'))
                y = batch.get('future_target')
            
            # Obter predições
            prediction_output = model(model_batch)
            
            # Para cada amostra no batch
            for i in range(min(len(x), num_samples - len(samples))):
                # Extrair parâmetros da distribuição Student-T
                loc = prediction_output.loc[i].cpu().numpy()
                scale = prediction_output.scale[i].cpu().numpy()
                df = prediction_output.df[i].cpu().numpy()
                
                # Calcular intervalos de confiança
                # Criar distribuição Student-T
                dist = StudentT(loc=torch.tensor(loc), scale=torch.tensor(scale), df=torch.tensor(df))
                
                # Calcular quantis para intervalos de confiança
                lower_95 = dist.icdf(torch.tensor(0.025)).cpu().numpy()
                upper_95 = dist.icdf(torch.tensor(0.975)).cpu().numpy()
                
                # Guardar os dados para visualização
                samples.append({
                    'x': x[i].cpu().numpy(),
                    'y': y[i].cpu().numpy(),
                    'loc': loc,
                    'lower_95': lower_95,
                    'upper_95': upper_95
                })
    
    # Visualizar as predições
    fig, axs = plt.subplots(len(samples), 1, figsize=(12, 5*len(samples)))
    if len(samples) == 1:
        axs = [axs]
    
    for i, sample in enumerate(samples):
        # Obter dados
        x_data = sample['x'][:, 0]  # Assumindo que a primeira feature é o target
        y_data = sample['y']
        loc = sample['loc']
        lower_95 = sample['lower_95']
        upper_95 = sample['upper_95']
        
        # Plotar série histórica
        axs[i].plot(range(len(x_data)), x_data, 'b-', label='Histórico')
        
        # Plotar valores reais
        forecast_start = len(x_data)
        axs[i].plot(range(forecast_start, forecast_start + len(y_data)), y_data, 'g-', label='Real')
        
        # Plotar previsão e intervalo de confiança
        axs[i].plot(range(forecast_start, forecast_start + len(loc)), loc, 'r-', label='Previsão')
        axs[i].fill_between(
            range(forecast_start, forecast_start + len(loc)),
            lower_95, upper_95,
            color='r', alpha=0.2, label='IC 95%'
        )
        
        # Configurar gráfico
        axs[i].set_title(f'Amostra {i+1}: Previsão com Intervalo de Confiança')
        axs[i].set_xlabel('Tempo')
        axs[i].set_ylabel('Valor')
        axs[i].legend()
        axs[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return samples

# Visualizar predições no conjunto de teste
samples = visualize_predictions_with_uncertainty(pl_module, test_loader, num_samples=3)

### 6.3 Análise de Métricas Bayesianas

In [ ]:
# Importar bibliotecas necessárias
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Função para calcular métricas de incerteza
def calculate_uncertainty_metrics(samples):
    results = []
    
    for i, sample in enumerate(samples):
        # Obter dados
        y_true = sample['y']
        y_pred = sample['loc']
        lower_95 = sample['lower_95']
        upper_95 = sample['upper_95']
        
        # Calcular métricas básicas
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        # Calcular largura média do intervalo de confiança
        ci_width = np.mean(upper_95 - lower_95)
        
        # Calcular cobertura do intervalo de confiança (% de pontos dentro do IC)
        in_interval = np.logical_and(y_true >= lower_95, y_true <= upper_95)
        coverage = np.mean(in_interval) * 100
        
        # Calcular CRPS (Continuous Ranked Probability Score) aproximado
        # Simplificação: usamos apenas média e desvio padrão para aproximar
        scale = (upper_95 - lower_95) / (2 * 1.96)  # Aproximação do desvio padrão
        crps_approx = np.mean(scale * (np.sqrt(2/np.pi) - 2 * norm.pdf((y_true - y_pred) / scale) - 
                                      (y_true - y_pred) / scale * (2 * norm.cdf((y_true - y_pred) / scale) - 1)))
        
        results.append({
            'Sample': i+1,
            'MAE': mae,
            'RMSE': rmse,
            'CI Width': ci_width,
            'Coverage (%)': coverage,
            'CRPS': crps_approx
        })
    
    # Converter para DataFrame
    metrics_df = pd.DataFrame(results)
    
    # Adicionar média
    metrics_df.loc['Mean'] = metrics_df.mean()
    metrics_df.loc['Mean', 'Sample'] = 'Mean'
    
    return metrics_df

# Calcular métricas para as amostras visualizadas
try:
    from scipy.stats import norm
    metrics_df = calculate_uncertainty_metrics(samples)
    print(metrics_df)
except Exception as e:
    print(f"Erro ao calcular métricas: {e}")

## 7. Salvamento do Modelo e Exportação

Vamos salvar o modelo treinado para uso posterior.

### 7.1 Salvamento do Modelo

In [ ]:
# Criar diretório para o modelo
model_dir = os.path.join(OUTPUT_DIR, "model")
os.makedirs(model_dir, exist_ok=True)

# Salvar o modelo completo (Lightning Module)
model_path = os.path.join(model_dir, "crypto_moirai_moe_bayesian.pt")
torch.save(pl_module.state_dict(), model_path)

# Salvar apenas o modelo base (sem o Lightning Module)
base_model_path = os.path.join(model_dir, "crypto_moirai_moe_bayesian_base.pt")
torch.save(model.state_dict(), base_model_path)

print(f"Modelo salvo em {model_path}")
print(f"Modelo base salvo em {base_model_path}")

### 7.2 Salvamento da Configuração

In [ ]:
# Salvar a configuração utilizada
config_path = os.path.join(model_dir, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f"Configuração salva em {config_path}")

### 7.3 Exportação de Artefatos para Download

In [ ]:
# Comprimir artefatos importantes para download
import zipfile
import glob

# Criar arquivo ZIP com os artefatos principais
zip_path = os.path.join(BASE_DIR, "crypto_moirai_moe_bayesian_artifacts.zip")
with zipfile.ZipFile(zip_path, "w") as zipf:
    # Adicionar modelo e configuração
    zipf.write(model_path, os.path.basename(model_path))
    zipf.write(base_model_path, os.path.basename(base_model_path))
    zipf.write(config_path, os.path.basename(config_path))
    
    # Adicionar logs
    for log_file in glob.glob(os.path.join(LOG_DIR, "**/*.csv"), recursive=True):
        zipf.write(log_file, os.path.join("logs", os.path.basename(log_file)))
    
    # Adicionar gráficos
    for plot_file in glob.glob(os.path.join(PLOT_DIR, "**/*.png"), recursive=True):
        zipf.write(plot_file, os.path.join("plots", os.path.basename(plot_file)))

print(f"Artefatos comprimidos em {zip_path}")
print(f"Faça o download deste arquivo para uso posterior.")

## Resumo e Próximos Passos

Neste notebook, realizamos o fine-tuning do modelo Moirai-MoE com componentes bayesianos para previsão de séries temporais de criptomoedas usando o repositório uni2ts. Os principais passos foram:

1. Configuração do ambiente Kaggle
2. Download e instalação do repositório uni2ts
3. Preparação dos dados de criptomoedas da Binance
4. Configuração do modelo Moirai-MoE com componentes bayesianos
5. Treinamento e monitoramento do modelo usando o `MoiraiBayesianLightningModule` existente no projeto
6. Avaliação do modelo e visualização de resultados com intervalos de confiança
7. Salvamento do modelo e exportação de artefatos

### Benefícios de Usar a Implementação Existente

Neste notebook, utilizamos a classe `MoiraiBayesianLightningModule` do projeto uni2ts em vez de redefinir nossa própria implementação. Isso traz vários benefícios:

1. **Consistência com o projeto**: Garantimos que nosso treinamento segue exatamente a mesma lógica do projeto original.
2. **Redução de bugs**: Evitamos possíveis erros ao reimplementar uma lógica já testada e validada.
3. **Manutenção facilitada**: Se o projeto original for atualizado, podemos facilmente incorporar essas melhorias.
4. **Reprodutibilidade**: Os resultados são mais consistentes com os benchmarks oficiais.

### Próximos Passos

Para continuar o desenvolvimento do modelo, você pode considerar:

1. **Ajuste de Hiperparâmetros**: Experimente diferentes configurações para melhorar o desempenho do modelo.
2. **Adicionar Mais Ativos**: Expanda o conjunto de dados com mais criptomoedas para melhorar a generalização.
3. **Adicionar Features Externas**: Incorpore indicadores econômicos, sentimento de mercado, ou outros dados relevantes.
4. **Otimização do Modelo**: Experimente diferentes arquiteturas e componentes no modelo Moirai-MoE.
5. **Deploy do Modelo**: Implemente o modelo em produção para previsão contínua.

### Referências

- [Repositório uni2ts](https://github.com/waldefran/uni2ts)
- [Documentação do PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
- [Tutorial de Fine-tuning no Kaggle](https://github.com/waldefran/uni2ts/blob/main/KAGGLE_FINETUNING_TUTORIAL.md)